# Official workload motivation
The application-level motivation uses the same official SWE-Bench Lite and Terminal-Bench task records as the main evaluation. Historical files such as `motivation_optimization_history.csv` and `motivation_runtime_comparison.csv` are retained only as provenance; they are not used to produce this figure.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)

def official_summary():
    frames = []
    for backend in ('deepseek_harness', 'codex', 'official'):
        path = RESULTS / backend / 'official_token_summary.csv'
        if path.exists():
            frames.append(pd.read_csv(path))
    if not frames:
        return pd.DataFrame(columns=['suite', 'scale', 'mode', 'harness_backend', 'total_tokens_mean'])
    return pd.concat(frames, ignore_index=True).drop_duplicates()

df = official_summary()
if not df.empty:
    view = (df.groupby(['suite', 'scale', 'mode'], as_index=False)['total_tokens_mean'].mean())
    fig, ax = plt.subplots(figsize=(6.8, 3.0), dpi=300)
    for (suite, mode), group in view.groupby(['suite', 'mode']):
        group = group.sort_values('scale')
        ax.plot(group['scale'], group['total_tokens_mean'], marker='o', linewidth=1.0, label=f'{suite}:{mode}')
    ax.set_xlabel('Task scale')
    ax.set_ylabel('Total tokens (mean)')
    ax.legend(fontsize=6, ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Motivation-Official-Token.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Motivation-Official-Token.pdf', bbox_inches='tight')
else:
    print('No official summary found; run experiments/scripts/bench_official_tasks.py first.')
print('official workload rows:', len(df))
# Compatibility names: motivation_optimization_history.csv, motivation_runtime_comparison.csv
# The canonical input is official_token_summary.csv, not a synthetic trajectory.
# chain: official suite -> external harness -> causal recovery -> token summary
